# 48. Why HGB with the spectral feature gains seven onset-anchored catches: a mechanism diagnostic (BTC spot)

In notebook 47 I found, for the 4-hour event definition and the same 800-alert budgets, 9 (HGB_Base) versus 16 (HGB_HS) pre-onset event catches in the onset-anchored 30–240 minute window, but 13 versus 12 in the breach-anchored window. Here I ask why the ranking changes and separate four mechanisms:

1. Event-level timing: which 8 events HGB_HS gains and which 1 it loses, and which alerts come before the price peak rather than only before the 1% onset.
2. Endpoint and window geometry: whether a new alert falls before the breach-anchored window or misses it for another reason.
3. Score versus scheduler: whether HGB_Base was below its threshold at the same timestamp, or whether the 240-minute cooldown or the monthly cap kept it from alerting.
4. Spectral model response: whether the HGB_HS score stays above threshold when I replace `z_hs` by its fixed 2021 training median and hold the two other features constant; this is a model-input perturbation, not a causal counterfactual.

The first four sections reproduce notebook 47 (BTC spot, 2021 training, 2022 calibration, 2023 to April 2026 development). Status: this is a forensic, exploratory, post hoc diagnostic in which I train no new selected model and leave the preregistered prospective test unchanged. I spotted the 8-versus-1 difference after inspecting several models and anchors, so descriptive event bootstraps, per-event explanations and unadjusted significance do not establish incremental predictability or generalization. Results folder: `spectral_onset_mechanism_diagnostic_v1`.

In [1]:
from pathlib import Path
import sys, subprocess, json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

try:
    import pyarrow
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])

try:
    from numba import njit
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab. Configure ROOT manually to run with local files.')

ROOT = Path('/content/drive/MyDrive/btc_lead_cost_results_77months_spot')
CACHE = ROOT/'cache'
REPRO47 = ROOT/'spectral_nonlinear_interactions_v1'
OUT = ROOT/'spectral_onset_mechanism_diagnostic_v1'
OUT.mkdir(parents=True, exist_ok=True)
NS = 60*10**9
T0 = pd.Timestamp('2021-01-01', tz='UTC').value
T1 = pd.Timestamp('2022-01-01', tz='UTC').value
T2 = pd.Timestamp('2023-01-01', tz='UTC').value
E0 = pd.Timestamp('2023-01-01', tz='UTC').value
E1 = pd.Timestamp('2026-05-01', tz='UTC').value
LEAD_MIN, LEAD_MAX = 30, 240
COOLDOWN = 240
BUDGET_MONTH, BUDGET_YEAR = 20, 240
SPEC = {
    'notebook': '47', 'status': 'exploratory_posthoc',
    'source': 'BTC spot frozen cache', 'train': '2021',
    'target': '4% breach 30-240 min ahead',
    'threshold_calibration': '2022_unlabeled',
    'evaluation': '2023-01 through 2026-04',
    'models': ['Base_Logit','HS_Linear','HS_Interactions','HGB_Base','HGB_HS'],
    'interaction_features': ['z_hs*z_vol','z_hs*(tsr-0.5)','z_hs^2'],
    'clip': 6.0,
    'HGB': dict(max_iter=100, max_leaf_nodes=7, learning_rate=0.05,
                min_samples_leaf=100, l2_regularization=5.0, max_bins=127,
                early_stopping=False, random_state=20261147,
                class_weight='balanced_via_sample_weight'),
    'comparisons': [('HS_Linear','HS_Interactions'),('HGB_Base','HGB_HS')],
    'lead_minutes': [30,240], 'monthly_cap': 20,
    'annual_cap': 240, 'cooldown_minutes': 240,
    'primary': '4h breach-anchored pre-onset catches per 100 alerts',
    'secondary': '4h within-RV-decile pre-onset AUC; onset-anchor and H=1440 descriptive',
    'stop_go': {'within_rv_decile_auc_gain':0.02,
                'pre_catches_per_100_alerts_gain':0.5},
    'selection_note': 'Do not choose a winner from these repeatedly inspected development outcomes',
    'no_holdout_or_prospective_inputs': True
}
(OUT/'47_reproduction_SPEC.json').write_text(json.dumps(SPEC,indent=2))
print('Output:',OUT)

Mounted at /content/drive
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/spectral_onset_mechanism_diagnostic_v1


## 1. Spot data and the event catalog

I use only BTC spot one-minute parquets and existing non-anticipating features; `z_hs` is the notebook 01 FFT statistic, not a new spectral construction, and future event times enter only as training labels, never as predictors.

In [2]:
for p in [CACHE/'BTCUSDT_1m_regularized.parquet', CACHE/'btc_signals_1m.parquet', CACHE/'DATA_SOURCE.json']:
    if not p.is_file():
        raise FileNotFoundError(f'Required frozen spot cache missing: {p}')
source = json.loads((CACHE/'DATA_SOURCE.json').read_text())
assert source.get('venue')=='spot', 'The cache must be BTC SPOT; no futures substitute.'
px = pd.read_parquet(CACHE/'BTCUSDT_1m_regularized.parquet').sort_index()
sg = pd.read_parquet(CACHE/'btc_signals_1m.parquet', columns=['z_vol','tsr','z_hs']).sort_index()
for d in (px, sg):
    assert isinstance(d.index, pd.DatetimeIndex) and d.index.tz is not None
    d.index = d.index.tz_convert('UTC')
assert px.index.equals(sg.index), 'Price/signal minute indexes differ'
T = px.index.asi8
assert np.all(np.diff(T)==NS), 'Index is not a complete one-minute grid'
close = px.close.to_numpy(dtype=float)
volume = px.volume.to_numpy(dtype=float)
X = sg[['z_vol','tsr','z_hs']].to_numpy(dtype=float)
valid = (np.isfinite(close)&(close>0)&np.isfinite(volume)&(volume>0)&
         np.isfinite(X).all(axis=1)&(X[:,1]>=0)&(X[:,1]<=1))
print('rows',len(T),'train valid',int((valid&(T>=T0)&(T<T1)).sum()),'eval valid',int((valid&(T>=E0)&(T<E1)).sum()))

def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory: continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi],'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    if not rows: raise RuntimeError('Event catalog empty')
    return pd.DataFrame(rows)

CAT = {h:price_only_catalog(close,volume,px.index,H=h) for h in (240,1440)}
EVAL = {h:CAT[h].loc[(CAT[h].breach_utc.astype('int64')>=E0+LEAD_MAX*NS)&
                             (CAT[h].breach_utc.astype('int64')<E1)].copy().reset_index(drop=True)
        for h in (240,1440)}
# Onset-anchored evaluation needs a complete onset-240 ... onset-30 window.
EVAL_ON = {h: EVAL[h].loc[EVAL[h].onset_1pct_utc.astype('int64') >= E0 + LEAD_MAX*NS].reset_index(drop=True)
           for h in (240,1440)}
# Mandatory guard against the reference 4-hour development event count
assert len(EVAL[240])==83, f'Expected 83 development events at H=240, found {len(EVAL[240])}'
assert len(EVAL[1440])==218, f'Expected 218 events at H=1440, found {len(EVAL[1440])}'
# Optional: exact timestamp comparison with notebook 2
ref = ROOT/'event_definition_eligibility_audit'/'price_only_event_catalog.csv'
if ref.is_file():
    q = pd.read_csv(ref)
    for c in ('peak_utc','onset_1pct_utc','breach_utc','confirmed_utc'):
        a = pd.to_datetime(q[c],utc=True).astype('int64').to_numpy()
        b = CAT[240][c].astype('int64').to_numpy()
        assert np.array_equal(a,b), f'Notebook 2 mismatch for {c}'
    print('PASS: notebook-2 event timestamps reproduced.')

# Labels from the next qualifying event timestamp (same searchsorted convention as notebook 41).
def future_label(T, events_ns):
    z=np.sort(np.asarray(events_ns,dtype=np.int64))
    positions=np.searchsorted(z,T+LEAD_MIN*NS,side='left')
    y=np.zeros(len(T),dtype=np.int8)
    good=positions<len(z)
    y[good]=(z[positions[good]]<=T[good]+LEAD_MAX*NS).astype(np.int8)
    return y

cat4=CAT[240]
y_breach = future_label(T,cat4.breach_utc.astype('int64').to_numpy())
y_onset = future_label(T,cat4.onset_1pct_utc.astype('int64').to_numpy())
# No retrospective classification rule is used as a runtime signal. During onset training,
# remove confirmed-event in-progress minutes from the negative training examples.
in_progress = np.zeros(len(T),bool)
for e in cat4.itertuples(index=False):
    a=np.searchsorted(T,e.onset_1pct_utc.value,'left')
    b=np.searchsorted(T,e.confirmed_utc.value,'right')
    in_progress[a:b]=True
TRAIN = valid &(T>=T0)&(T<T1-(LEAD_MAX+5)*NS)
TRAIN_ONSET = TRAIN & ~in_progress
CAL = valid &(T>=T1)&(T<T2)
DEV = valid &(T>=E0)&(T<E1-(LEAD_MAX+5)*NS)
print('2021 breach positive minutes:',int(y_breach[TRAIN].sum()))
print('2021 onset positive minutes:',int(y_onset[TRAIN_ONSET].sum()))
if not (y_breach[TRAIN].sum()>50 and y_onset[TRAIN_ONSET].sum()>50):
    raise RuntimeError('Insufficient training positives: inspect event labels and cache.')

rows 3329280 train valid 524332 eval valid 1750857
PASS: notebook-2 event timestamps reproduced.
2021 breach positive minutes: 18779
2021 onset positive minutes: 18771


## 2. Five fitted models, no tuning on development data

The logistic interaction arms add fixed polynomial products, and the HGB arms share one architecture and differ only in whether they include `z_hs`. I balance classes with training sample weights and disable HGB early stopping.

In [3]:
# Five arms. All use the same breach target and TRAIN mask.
# This isolates the predictor form, unlike comparisons that also change the target.
CLIP=6.0
zv=np.clip(X[:,0],-CLIP,CLIP)
ts=X[:,1]
hs=np.clip(X[:,2],-CLIP,CLIP)
# Fit the baseline M0 and the linear z_hs ablation on the untouched raw predictors.
X_INTERACT=np.column_stack([X[:,0],X[:,1],X[:,2],hs*zv,hs*(ts-.5),hs*hs])
assert np.isfinite(X_INTERACT[valid]).all(), 'Feature construction produced non-finite observations'

LOGIT_BASE=make_pipeline(StandardScaler(),LogisticRegression(
    C=0.2,class_weight='balanced',max_iter=400,solver='lbfgs',random_state=17))
LOGIT_HS=make_pipeline(StandardScaler(),LogisticRegression(
    C=0.2,class_weight='balanced',max_iter=400,solver='lbfgs',random_state=17))
LOGIT_INTER=make_pipeline(StandardScaler(),LogisticRegression(
    C=0.2,class_weight='balanced',max_iter=400,solver='lbfgs',random_state=17))
LOGIT_BASE.fit(X[TRAIN][:,:2],y_breach[TRAIN])
LOGIT_HS.fit(X[TRAIN],y_breach[TRAIN])
LOGIT_INTER.fit(X_INTERACT[TRAIN],y_breach[TRAIN])

# Keep baseline guard from notebook 46 / notebook 16 if present.
frozen=ROOT/'methods_definitions_v1'/'METHODS_FROZEN.json'
if frozen.is_file():
    f=json.loads(frozen.read_text())
    cfg=f['configs']['M0_VF_logit|4pct']
    assert np.allclose(LOGIT_BASE[-1].coef_.ravel(),cfg['coef'],rtol=1e-6,atol=1e-9), 'M0 coefficient mismatch'
    assert np.isclose(LOGIT_BASE[-1].intercept_[0],cfg['intercept'],rtol=1e-6,atol=1e-9), 'M0 intercept mismatch'
    print('PASS: frozen M0 logistic parameters')
else:
    print('Warning: frozen M0 file missing; the AUC and alert guards still apply')

# Small nonlinear model: exact same hyperparameters with vs without spectral input.
# HGB supports sample_weight in all relevant sklearn releases; avoid version-dependent class_weight.
_,cnt=np.unique(y_breach[TRAIN],return_counts=True)
n_pos=int(y_breach[TRAIN].sum());n_neg=int(TRAIN.sum()-n_pos)
assert n_pos>50 and n_neg>n_pos
train_w=np.where(y_breach[TRAIN]==1,(n_pos+n_neg)/(2*n_pos),(n_pos+n_neg)/(2*n_neg))
def fixed_hgb():
    return HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=7,
        learning_rate=0.05,min_samples_leaf=100,l2_regularization=5.0,
        max_bins=127,early_stopping=False,random_state=20261147)
HGB_BASE=fixed_hgb();HGB_HS=fixed_hgb()
HGB_BASE.fit(X[TRAIN][:,:2],y_breach[TRAIN],sample_weight=train_w)
HGB_HS.fit(X[TRAIN],y_breach[TRAIN],sample_weight=train_w)
MODELS={
 'Base_Logit':(LOGIT_BASE,'base'),
 'HS_Linear':(LOGIT_HS,'hs'),
 'HS_Interactions':(LOGIT_INTER,'interactions'),
 'HGB_Base':(HGB_BASE,'base'),
 'HGB_HS':(HGB_HS,'hs'),
}
print('Training count',int(TRAIN.sum()),'positive labels',int(n_pos))

# Predict in chunks, only for 2022 calibration + 2023-2026 development.
SCORE={}
mask_pred=CAL|DEV
indices=np.flatnonzero(mask_pred)
for name,(model,which) in MODELS.items():
    scores=np.full(len(T),np.nan,dtype=np.float64)
    for block in np.array_split(indices,max(1,int(np.ceil(len(indices)/180000)))):
        xx= X_INTERACT[block] if which=='interactions' else (X[block,:2] if which=='base' else X[block])
        scores[block]=model.predict_proba(xx)[:,1]
    assert np.isfinite(scores[mask_pred]).all(),f'{name}: nonfinite scores'
    SCORE[name]=scores
    print(name,'score computed on',len(indices),'2022+development minutes',flush=True)
# Free the largest temporary arrays; the score arrays and base inputs remain.
del X_INTERACT,zv,ts,hs

PASS: frozen M0 logistic parameters
Training count 524087 positive labels 18779
Base_Logit score computed on 2276212 2022+development minutes
HS_Linear score computed on 2276212 2022+development minutes
HS_Interactions score computed on 2276212 2022+development minutes
HGB_Base score computed on 2276212 2022+development minutes
HGB_HS score computed on 2276212 2022+development minutes


## 3. Calendar scheduler and unlabeled 2022 calibration

The scheduler code is the same as in notebook 46. I report exact alert counts and differences per 100 alerts, so no candidate gains credit for sending more alerts.

In [4]:
@njit(cache=False)
def calendar_core(t,elig,years,months,B,cap,cooldown_ns):
    out=np.empty(len(t),dtype=np.int64)
    n=0; month_count=0; year_count=0; last_m=-1; last_y=-1; last=-(2**62)
    for i in range(len(t)):
        if not elig[i]: continue
        yr=years[i]; mo=months[i]; ti=t[i]
        if yr!=last_y: year_count=0; last_y=yr
        if mo!=last_m: month_count=0; last_m=mo
        if month_count>=cap or year_count>=B or ti-last<cooldown_ns: continue
        out[n]=ti;n+=1;month_count+=1;year_count+=1;last=ti
    return out[:n].copy()

def scheduler(time_ns,score,threshold):
    tm=pd.to_datetime(time_ns,utc=True)
    yrs=np.asarray(tm.year,dtype=np.int64)
    months=np.asarray(tm.year*12+tm.month,dtype=np.int64)
    e=np.asarray(np.isfinite(score)&(score>=threshold),dtype=np.bool_)
    return calendar_core(np.asarray(time_ns,np.int64),e,yrs,months,BUDGET_YEAR,BUDGET_MONTH,COOLDOWN*NS)

def score_grid(sc):
    Q=np.unique(np.r_[0.,np.linspace(.10,.995,33),.997,.999,.9999,1.])
    return np.unique(np.r_[np.nextafter(sc.min(),-np.inf),np.quantile(sc,Q),np.nextafter(sc.max(),np.inf)])

def calibrate(time_ns,sc):
    rows=[]
    for th in score_grid(sc):
        count=len(scheduler(time_ns,sc,th))
        rows.append((float(th),count,int(np.count_nonzero(sc>=th))))
    d=pd.DataFrame(rows,columns=['threshold','cal_alerts','raw_candidates'])
    row=d[d.cal_alerts==d.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(row.threshold),int(row.cal_alerts),d

T_CAL=T[CAL];T_DEV=T[DEV]
THR={}; ALARMS={}; CAL_COUNTS={}
for name in MODELS:
    thr,ncal,_=calibrate(T_CAL,SCORE[name][CAL])
    THR[name]=thr;CAL_COUNTS[name]=ncal
    ALARMS[name]=scheduler(T_DEV,SCORE[name][DEV],thr)
    print(f'{name:14s} 2022 cal alerts={ncal:3d}  threshold={thr:.6g}  development alerts={len(ALARMS[name])}')
assert all(len(x)<=40*20 for x in ALARMS.values()),'Exceeded maximum 40-month 20-per-month budget'

Base_Logit     2022 cal alerts=240  threshold=0.669899  development alerts=800
HS_Linear      2022 cal alerts=240  threshold=0.671414  development alerts=800
HS_Interactions 2022 cal alerts=240  threshold=0.667268  development alerts=800
HGB_Base       2022 cal alerts=240  threshold=0.621315  development alerts=800
HGB_HS         2022 cal alerts=240  threshold=0.626659  development alerts=800


## 4. Pre-onset catches

The primary endpoint is the 4-hour breach-anchored judging window `[breach - 240, breach - 30]` with alerts before onset; the onset-anchored window and the 24-hour definition are secondary. These are different tasks, so a larger onset-anchored count is not an improvement on the breach-anchored endpoint.

In [5]:
def event_catches(events, alerts, anchor='breach'):
    alert=np.asarray(alerts,dtype=np.int64)
    rows=[]
    for e in events.itertuples(index=False):
        on=e.onset_1pct_utc.value;b=e.breach_utc.value
        anchor_ns=on if anchor=='onset' else b
        lo=anchor_ns-LEAD_MAX*NS; hi=anchor_ns-LEAD_MIN*NS
        l=np.searchsorted(alert,lo,side='left')
        r=np.searchsorted(alert,hi,side='right')
        eligible=alert[l:r]
        good=eligible[eligible<on]
        rows.append((e.event_id,len(good)>0,(on-int(good[0]))/NS if len(good) else np.nan))
    return pd.DataFrame(rows,columns=['event_id','hit','lead_to_onset_minutes'])

rows=[];HITS={}
for h in (240,1440):
    for name in MODELS:
        for anchor in ('breach','onset'):
            E = EVAL_ON[h] if anchor=='onset' else EVAL[h]
            hit=event_catches(E,ALARMS[name],anchor)
            HITS[(h,name,anchor)]=hit
            n=len(ALARMS[name]);k=int(hit.hit.sum())
            rows.append({'H':h,'model':name,'anchor':anchor,'events':len(E),
                         'alerts':n,'pre_catches':k,'pre_recall':k/len(E),
                         'pre_catches_per_100_alerts':100*k/n if n else np.nan,
                         'median_lead_minutes':hit.lead_to_onset_minutes.median(),
                         '2022_calibration_alerts':CAL_COUNTS[name]})
RESULTS=pd.DataFrame(rows)
from IPython.display import display
display(RESULTS)

,H,model,anchor,events,alerts,pre_catches,pre_recall,pre_catches_per_100_alerts,median_lead_minutes,2022_calibration_alerts
0,240,Base_Logit,breach,83,800,22,0.265060,2.750,25.0,240
1,240,Base_Logit,onset,83,800,15,0.180723,1.875,72.0,240
2,240,HS_Linear,breach,83,800,18,0.216867,2.250,42.5,240
3,240,HS_Linear,onset,83,800,13,0.156627,1.625,64.0,240
4,240,HS_Interactions,breach,83,800,14,0.168675,1.750,60.5,240
5,240,HS_Interactions,onset,83,800,14,0.168675,1.750,94.0,240
6,240,HGB_Base,breach,83,800,13,0.156627,1.625,27.0,240
7,240,HGB_Base,onset,83,800,9,0.108434,1.125,97.0,240
8,240,HGB_HS,breach,83,800,12,0.144578,1.500,65.5,240
9,240,HGB_HS,onset,83,800,16,0.192771,2.000,134.5,240


## 5. Reproduction and 83-event hit matrix

With the same trained models, pre-onset definitions, fixed 2022 threshold calibration and 800 alerts as in notebook 47, I reproduce the 9/16 onset-anchored and 13/12 breach-anchored catches and record in a per-event table the eight newly caught events and the one lost event.

In [6]:
# Strict reproduction guards before any mechanism analysis.
def n_hit(h, model, anchor):
    return int(RESULTS.query('H == @h and model == @model and anchor == @anchor').pre_catches.iloc[0])
assert len(ALARMS['HGB_Base']) == len(ALARMS['HGB_HS']) == 800
assert (n_hit(240,'HGB_Base','onset'),n_hit(240,'HGB_HS','onset')) == (9,16)
assert (n_hit(240,'HGB_Base','breach'),n_hit(240,'HGB_HS','breach')) == (13,12)
assert (n_hit(240,'Base_Logit','breach'),n_hit(240,'Base_Logit','onset')) == (22,15)
prior=REPRO47/'1_event_level.csv'
if prior.is_file():
    old=pd.read_csv(prior)
    for name in ('HGB_Base','HGB_HS'):
        for anch in ('breach','onset'):
            cur=n_hit(240,name,anch)
            assert cur == int(old.query('H == 240 and model == @name and anchor == @anch').pre_catches.iloc[0])
    print('PASS: HGB aggregate catches reproduce saved notebook 47')
else:
    print('Warning: saved notebook 47 aggregate unavailable; the hard-coded guards passed')

def qualifying_alerts(e, alert_ns, anchor='onset'):
    on=e.onset_1pct_utc.value;b=e.breach_utc.value
    fixed=on if anchor=='onset' else b
    lo=fixed-LEAD_MAX*NS; hi=fixed-LEAD_MIN*NS
    a=np.asarray(alert_ns,dtype=np.int64)
    left=np.searchsorted(a,lo,'left');right=np.searchsorted(a,hi,'right')
    z=a[left:right]
    return z[z<on]

def iso(ns):
    return pd.Timestamp(int(ns),tz='UTC').isoformat() if ns is not None and pd.notna(ns) else ''

E=EVAL_ON[240]
rows=[]
for e in E.itertuples(index=False):
    b0=qualifying_alerts(e,ALARMS['HGB_Base'],'onset')
    hs=qualifying_alerts(e,ALARMS['HGB_HS'],'onset')
    b0_b=qualifying_alerts(e,ALARMS['HGB_Base'],'breach')
    hs_b=qualifying_alerts(e,ALARMS['HGB_HS'],'breach')
    group='new_only' if len(hs) and not len(b0) else ('base_only' if len(b0) and not len(hs) else ('both' if len(b0) and len(hs) else 'neither'))
    rows.append(dict(event_id=e.event_id,peak_utc=e.peak_utc,onset_utc=e.onset_1pct_utc,
       breach_utc=e.breach_utc,onset_to_breach_min=int(e.onset_to_breach_minutes),
       onset_group=group,base_onset_hit=bool(len(b0)),hs_onset_hit=bool(len(hs)),
       base_breach_hit=bool(len(b0_b)),hs_breach_hit=bool(len(hs_b)),
       base_first_onset_alert_utc=iso(b0[0]) if len(b0) else '',
       hs_first_onset_alert_utc=iso(hs[0]) if len(hs) else '',
       base_onset_lead_min=(e.onset_1pct_utc.value-b0[0])/NS if len(b0) else np.nan,
       hs_onset_lead_min=(e.onset_1pct_utc.value-hs[0])/NS if len(hs) else np.nan,
       hs_alert_before_price_peak=bool(hs[0]<e.peak_utc.value) if len(hs) else False,
       base_alert_before_price_peak=bool(b0[0]<e.peak_utc.value) if len(b0) else False,
       hs_alert_outside_breach_window=bool(len(hs) and not (e.breach_utc.value-LEAD_MAX*NS<=hs[0]<=e.breach_utc.value-LEAD_MIN*NS)),
       hs_alert_to_breach_min=(e.breach_utc.value-hs[0])/NS if len(hs) else np.nan,
    ))
EVENTS=pd.DataFrame(rows)
EVENTS.to_csv(OUT/'1_onset_event_hit_matrix_4h.csv',index=False)
from IPython.display import display
print(EVENTS.onset_group.value_counts().to_string())
assert int((EVENTS.onset_group=='new_only').sum())==8
assert int((EVENTS.onset_group=='base_only').sum())==1
assert int((EVENTS.onset_group=='both').sum())==8
assert int((EVENTS.onset_group=='neither').sum())==66
print('PASS: 8 new-only, 1 base-only, 8 both, 66 neither = 83 events')
display(EVENTS[EVENTS.onset_group.isin(['new_only','base_only'])].sort_values('onset_utc'))


PASS: HGB aggregate catches reproduce saved notebook 47
onset_group
neither      66
new_only      8
both          8
base_only     1
PASS: 8 new-only, 1 base-only, 8 both, 66 neither = 83 events


,event_id,peak_utc,onset_utc,breach_utc,onset_to_breach_min,onset_group,base_onset_hit,hs_onset_hit,base_breach_hit,hs_breach_hit,base_first_onset_alert_utc,hs_first_onset_alert_utc,base_onset_lead_min,hs_onset_lead_min,hs_alert_before_price_peak,base_alert_before_price_peak,hs_alert_outside_breach_window,hs_alert_to_breach_min
3,190,2023-03-09 16:15:00+00:00,2023-03-09 17:02:00+00:00,2023-03-09 20:11:00+00:00,189,new_only,False,True,False,False,,2023-03-09T13:36:00+00:00,NaN,206.0,True,False,True,395.0
9,196,2023-06-05 14:47:00+00:00,2023-06-05 15:25:00+00:00,2023-06-05 17:09:00+00:00,104,new_only,False,True,True,True,,2023-06-05T14:36:00+00:00,NaN,49.0,True,False,False,153.0
15,202,2023-11-09 14:53:00+00:00,2023-11-09 15:23:00+00:00,2023-11-09 16:35:00+00:00,72,base_only,True,False,True,False,2023-11-09T12:48:00+00:00,,155.0,NaN,False,True,False,NaN
17,204,2024-01-03 09:39:00+00:00,2024-01-03 11:07:00+00:00,2024-01-03 12:00:00+00:00,53,new_only,False,True,False,False,,2024-01-03T07:46:00+00:00,NaN,201.0,True,False,True,254.0
42,229,2024-08-05 16:29:00+00:00,2024-08-05 16:34:00+00:00,2024-08-05 19:07:00+00:00,153,new_only,False,True,True,True,,2024-08-05T15:33:00+00:00,NaN,61.0,True,False,False,214.0
50,237,2024-12-05 18:40:00+00:00,2024-12-05 18:59:00+00:00,2024-12-05 22:26:00+00:00,207,new_only,False,True,False,False,,2024-12-05T15:29:00+00:00,NaN,210.0,True,False,True,417.0
59,246,2025-03-03 13:43:00+00:00,2025-03-03 14:35:00+00:00,2025-03-03 14:56:00+00:00,21,new_only,False,True,False,True,,2025-03-03T11:02:00+00:00,NaN,213.0,True,False,False,234.0
62,249,2025-04-02 20:16:00+00:00,2025-04-02 20:22:00+00:00,2025-04-02 21:09:00+00:00,47,new_only,False,True,True,True,,2025-04-02T17:53:00+00:00,NaN,149.0,True,False,False,196.0
76,263,2026-02-02 00:38:00+00:00,2026-02-02 01:09:00+00:00,2026-02-02 03:34:00+00:00,145,new_only,False,True,True,True,,2026-02-01T23:56:00+00:00,NaN,73.0,True,False,False,218.0


## 6. Each rescued onset event

For each newly caught event I locate the actual HGB_HS alert, its lead relative to onset and breach, whether the price peak had already passed, the spectral feature value at that minute, each model's score against its own frozen threshold, and whether its own budget or cooldown blocked the baseline scheduler. I do not compare raw probabilities across models, because their score calibrations differ.

In [7]:
def n_this_month_before(alerts, t_ns):
    dt=pd.Timestamp(int(t_ns),tz='UTC')
    lo=pd.Timestamp(year=dt.year,month=dt.month,day=1,tz='UTC').value
    a=np.asarray(alerts,np.int64)
    return int(np.searchsorted(a,t_ns,side='left')-np.searchsorted(a,lo,side='left'))

def latest_prior(alerts,t_ns):
    a=np.asarray(alerts,np.int64)
    k=np.searchsorted(a,t_ns,side='left')-1
    return int(a[k]) if k>=0 else None

def scheduler_reason(model,alert_ns):
    idx=np.searchsorted(T,alert_ns)
    if idx>=len(T) or T[idx]!=alert_ns or not DEV[idx] or not np.isfinite(SCORE[model][idx]):
        return 'outside_scored_valid_grid'
    if SCORE[model][idx]<THR[model]:
        return 'below_own_threshold'
    previous=latest_prior(ALARMS[model],alert_ns)
    if previous is not None and alert_ns-previous<COOLDOWN*NS:
        return 'blocked_by_own_cooldown'
    if n_this_month_before(ALARMS[model],alert_ns)>=BUDGET_MONTH:
        return 'monthly_alert_budget_exhausted'
    return 'threshold_met_but_not_alerted_CHECK'

# The 2021 median is fixed independently of 2023-2026 outcomes.
TRAIN_HS_MEDIAN=float(np.median(X[TRAIN,2]))
TRAIN_HS_Q10,TRAIN_HS_Q90=np.quantile(X[TRAIN,2],[.1,.9])
assert np.isfinite(TRAIN_HS_MEDIAN)

def hs_spectral_perturbation(t_ns):
    ix=np.searchsorted(T,t_ns)
    x_true=X[ix].reshape(1,-1).copy()
    x_cf=x_true.copy();x_cf[0,2]=TRAIN_HS_MEDIAN
    observed=float(HGB_HS.predict_proba(x_true)[0,1])
    neutral=float(HGB_HS.predict_proba(x_cf)[0,1])
    return observed,neutral,observed-neutral

out=[]
for e in E.itertuples(index=False):
    g=EVENTS.loc[EVENTS.event_id==e.event_id,'onset_group'].iloc[0]
    if g not in ('new_only','base_only'): continue
    # For new-only use HGB_HS actual alert, for base-only use HGB_Base actual alert.
    model='HGB_HS' if g=='new_only' else 'HGB_Base'
    rival='HGB_Base' if g=='new_only' else 'HGB_HS'
    chosen=qualifying_alerts(e,ALARMS[model],'onset')
    assert len(chosen)>0
    t=int(chosen[0]);idx=np.searchsorted(T,t)
    prior=latest_prior(ALARMS[rival],t)
    hscore,nscore,uplift=hs_spectral_perturbation(t)
    # Use a local index slice for speed; no future data enters a model.
    left=np.searchsorted(T,e.onset_1pct_utc.value-LEAD_MAX*NS)
    right=np.searchsorted(T,e.onset_1pct_utc.value-LEAD_MIN*NS,side='right')
    out.append(dict(event_id=e.event_id,group=g,peak_utc=e.peak_utc,
      onset_utc=e.onset_1pct_utc,breach_utc=e.breach_utc,
      onset_to_breach_min=int(e.onset_to_breach_minutes),
      chosen_model=model,chosen_alert_utc=iso(t),
      alert_to_onset_min=(e.onset_1pct_utc.value-t)/NS,
      alert_to_breach_min=(e.breach_utc.value-t)/NS,
      alert_to_peak_min=(e.peak_utc.value-t)/NS,
      actually_before_peak=bool(t<e.peak_utc.value),
      inside_breach_window=bool(e.breach_utc.value-LEAD_MAX*NS<=t<=e.breach_utc.value-LEAD_MIN*NS),
      z_hs=float(X[idx,2]),z_vol=float(X[idx,0]),taker_sell_ratio=float(X[idx,1]),
      base_score=float(SCORE['HGB_Base'][idx]),base_threshold=float(THR['HGB_Base']),
      hs_score=float(SCORE['HGB_HS'][idx]),hs_threshold=float(THR['HGB_HS']),
      base_score_above_threshold=bool(SCORE['HGB_Base'][idx]>=THR['HGB_Base']),
      hs_score_above_threshold=bool(SCORE['HGB_HS'][idx]>=THR['HGB_HS']),
      rival_no_alert_reason=scheduler_reason(rival,t),
      rival_month_alert_count_before=n_this_month_before(ALARMS[rival],t),
      rival_last_alert_distance_min=(t-prior)/NS if prior is not None else np.nan,
      rival_candidate_minutes_in_onset_window=int((np.isfinite(SCORE[rival][left:right])&
            (SCORE[rival][left:right]>=THR[rival])&DEV[left:right]).sum()),
      hs_model_score_at_actual_zhs=hscore,
      hs_model_score_at_training_median_zhs=nscore,
      hs_model_spectral_score_uplift=uplift,
      hs_counterfactual_above_hs_threshold=bool(nscore>=THR['HGB_HS']),
    ))
AUDIT=pd.DataFrame(out).sort_values(['group','onset_utc'])
AUDIT.to_csv(OUT/'2_discordant_event_alert_mechanism.csv',index=False)
print('2021 training z_hs median',TRAIN_HS_MEDIAN,'quantiles',TRAIN_HS_Q10,TRAIN_HS_Q90)
print('Rival non-alert explanations:')
print(AUDIT.groupby(['group','rival_no_alert_reason']).size().to_string())
display(AUDIT[['event_id','group','onset_utc','chosen_alert_utc','alert_to_onset_min','alert_to_breach_min',
  'alert_to_peak_min','actually_before_peak','inside_breach_window','z_hs','z_vol',
  'rival_no_alert_reason','rival_month_alert_count_before',
  'hs_model_spectral_score_uplift','hs_counterfactual_above_hs_threshold']])


2021 training z_hs median -0.07467103749513626 quantiles -1.236089539527893 1.39021372795105
Rival non-alert explanations:
group      rival_no_alert_reason  
base_only  below_own_threshold        1
new_only   below_own_threshold        5
           blocked_by_own_cooldown    3


,event_id,group,onset_utc,chosen_alert_utc,alert_to_onset_min,alert_to_breach_min,alert_to_peak_min,actually_before_peak,inside_breach_window,z_hs,z_vol,rival_no_alert_reason,rival_month_alert_count_before,hs_model_spectral_score_uplift,hs_counterfactual_above_hs_threshold
2,202,base_only,2023-11-09 15:23:00+00:00,2023-11-09T12:48:00+00:00,155.0,227.0,125.0,True,True,1.085575,1.204872,below_own_threshold,20,0.000578,False
0,190,new_only,2023-03-09 17:02:00+00:00,2023-03-09T13:36:00+00:00,206.0,395.0,159.0,True,False,0.132111,0.799446,below_own_threshold,16,0.000000,True
1,196,new_only,2023-06-05 15:25:00+00:00,2023-06-05T14:36:00+00:00,49.0,153.0,11.0,True,True,-0.115453,0.720154,below_own_threshold,7,0.000000,True
3,204,new_only,2024-01-03 11:07:00+00:00,2024-01-03T07:46:00+00:00,201.0,254.0,113.0,True,False,1.830160,0.383967,below_own_threshold,5,0.096494,False
4,229,new_only,2024-08-05 16:34:00+00:00,2024-08-05T15:33:00+00:00,61.0,214.0,56.0,True,True,2.480879,2.529475,blocked_by_own_cooldown,17,-0.027146,True
5,237,new_only,2024-12-05 18:59:00+00:00,2024-12-05T15:29:00+00:00,210.0,417.0,191.0,True,False,1.684690,1.931202,blocked_by_own_cooldown,7,-0.008703,True
6,246,new_only,2025-03-03 14:35:00+00:00,2025-03-03T11:02:00+00:00,213.0,234.0,161.0,True,True,0.100343,0.760213,below_own_threshold,5,0.000000,True
7,249,new_only,2025-04-02 20:22:00+00:00,2025-04-02T17:53:00+00:00,149.0,196.0,143.0,True,True,1.064948,0.427557,below_own_threshold,2,0.070425,False
8,263,new_only,2026-02-02 01:09:00+00:00,2026-02-01T23:56:00+00:00,73.0,218.0,42.0,True,True,1.429423,2.012668,blocked_by_own_cooldown,5,-0.002869,True


## 7. Signal changes versus calendar-scheduler changes

All models emit 800 alerts but possibly at different minutes. Here I tabulate month-by-month alert overlap and check whether the eight newly caught events concentrate in a few months. The spectral median replacement is only a local model-response diagnostic, since it can move a model off its learned joint feature distribution, and is not an alternative full policy evaluation.

In [8]:
months=pd.period_range('2023-01','2026-04',freq='M')
ma=[]
for month in months:
    start=pd.Timestamp(month.start_time,tz='UTC').value
    stop=pd.Timestamp((month+1).start_time,tz='UTC').value
    a=ALARMS['HGB_Base'];b=ALARMS['HGB_HS']
    aa=a[(a>=start)&(a<stop)];bb=b[(b>=start)&(b<stop)]
    gain=EVENTS[(EVENTS.onset_group=='new_only')&(EVENTS.onset_utc>=pd.Timestamp(start,tz='UTC'))&
                (EVENTS.onset_utc<pd.Timestamp(stop,tz='UTC'))]
    ma.append(dict(month=str(month),base_alerts=len(aa),hs_alerts=len(bb),
      exact_timestamp_overlap=len(np.intersect1d(aa,bb)),new_onset_hits=len(gain)))
MONTHS=pd.DataFrame(ma)
MONTHS.to_csv(OUT/'3_monthly_alert_overlap.csv',index=False)
print('Same-minute alert overlap overall:',len(np.intersect1d(ALARMS['HGB_Base'],ALARMS['HGB_HS'])),
      'of',len(ALARMS['HGB_Base']),'alerts per model')
print('Rescued events before peak:',int(AUDIT.query('group=="new_only"').actually_before_peak.sum()),'out of 8')
print('Rescued events with alert outside the breach-anchored window:',
      int((~AUDIT.query('group=="new_only"').inside_breach_window).sum()),'out of 8')
print('Rescued events with baseline score below its own threshold at new alert:',
      int((~AUDIT.query('group=="new_only"').base_score_above_threshold).sum()),'out of 8')
print('Rescued events whose HS model would not cross threshold at its own alert with z_hs fixed to 2021 median:',
      int((~AUDIT.query('group=="new_only"').hs_counterfactual_above_hs_threshold).sum()),'out of 8')
display(MONTHS[MONTHS.new_onset_hits>0])


Same-minute alert overlap overall: 102 of 800 alerts per model
Rescued events before peak: 8 out of 8
Rescued events with alert outside the breach-anchored window: 3 out of 8
Rescued events with baseline score below its own threshold at new alert: 5 out of 8
Rescued events whose HS model would not cross threshold at its own alert with z_hs fixed to 2021 median: 2 out of 8


,month,base_alerts,hs_alerts,exact_timestamp_overlap,new_onset_hits
2,2023-03,20,20,1,1
5,2023-06,20,20,6,1
12,2024-01,20,20,5,1
19,2024-08,20,20,1,1
23,2024-12,20,20,0,1
26,2025-03,20,20,5,1
27,2025-04,20,20,0,1
37,2026-02,20,20,0,1


## 8. Diagnosis summary

The primary breach-anchored screen of notebook 47 failed, and these diagnostics describe one post hoc onset-anchored difference. Alerts before the price peak are stronger descriptive temporal evidence, since the 1% onset occurs after the peak. Rescued alerts outside the breach-anchored window point to an endpoint re-anchoring effect rather than performance on the breach-anchored task. A baseline below its own threshold means that the spectral-inclusive scoring shifted the alert ranking, a baseline blocked by budget or cooldown means that the scheduler also explains the difference, and both can hold at once. If fixing `z_hs` at the frozen median removes the threshold crossing at an actual alert, the spectral feature has direct model influence there, which still does not show new information or event anticipation.

In [9]:
summary={
 'notebook':'48','status':'exploratory_posthoc_diagnostic_only',
 'source':'BTC spot frozen 47 inputs; models reconstructed from 2021',
 'model_types':['HGB_Base','HGB_HS'],
 'endpoint_primary_original':'4h breach-anchored pre-onset catches; 47 screen failed',
 'endpoint_diagnostic':'4h onset-anchored pre-onset catches; selected after observed gain',
 'total_event_count':len(E),
 'base_onset_catches':n_hit(240,'HGB_Base','onset'),
 'hs_onset_catches':n_hit(240,'HGB_HS','onset'),
 'base_breach_catches':n_hit(240,'HGB_Base','breach'),
 'hs_breach_catches':n_hit(240,'HGB_HS','breach'),
 'discordant_onset':EVENTS.onset_group.value_counts().to_dict(),
 'rescue_before_peak':int(AUDIT.query('group=="new_only"').actually_before_peak.sum()),
 'rescue_outside_breach_window':int((~AUDIT.query('group=="new_only"').inside_breach_window).sum()),
 'rescue_baseline_below_threshold':int((~AUDIT.query('group=="new_only"').base_score_above_threshold).sum()),
 'rescue_rival_nonalert_reasons':AUDIT.query('group=="new_only"').rival_no_alert_reason.value_counts().to_dict(),
 'rescue_spectral_median_counterfactual_below_threshold':int((~AUDIT.query('group=="new_only"').hs_counterfactual_above_hs_threshold).sum()),
 'same_minute_alert_overlap':int(len(np.intersect1d(ALARMS['HGB_Base'],ALARMS['HGB_HS']))),
 'data_snooping_note':'Exploratory forensic reading, no new policy developed or out-of-sample validation, no p-value confirmation.'
}
(OUT/'48_SUMMARY.json').write_text(json.dumps(summary,indent=2,default=str))
print(json.dumps(summary,indent=2,default=str))
print('Saved results to',OUT)


{
  "notebook": "48",
  "status": "exploratory_posthoc_diagnostic_only",
  "source": "BTC spot frozen 47 inputs; models reconstructed from 2021",
  "model_types": [
    "HGB_Base",
    "HGB_HS"
  ],
  "endpoint_primary_original": "4h breach-anchored pre-onset catches; 47 screen failed",
  "endpoint_diagnostic": "4h onset-anchored pre-onset catches; selected after observed gain",
  "total_event_count": 83,
  "base_onset_catches": 9,
  "hs_onset_catches": 16,
  "base_breach_catches": 13,
  "hs_breach_catches": 12,
  "discordant_onset": {
    "neither": 66,
    "new_only": 8,
    "both": 8,
    "base_only": 1
  },
  "rescue_before_peak": 8,
  "rescue_outside_breach_window": 3,
  "rescue_baseline_below_threshold": 5,
  "rescue_rival_nonalert_reasons": {
    "below_own_threshold": 5,
    "blocked_by_own_cooldown": 3
  },
  "rescue_spectral_median_counterfactual_below_threshold": 2,
  "same_minute_alert_overlap": 102,
  "data_snooping_note": "Exploratory forensic reading, no new policy devel